# CorePromoter TSS/PAS pretraining

This notebook creates a species-grouped TSS/PAS dataset, pretrains architecture recognition, recalibrates expression on the existing small libraries, and saves a frozen checkpoint for recursive design. It never overwrites `weights_CorePromoter_clean.pt`.

In [ ]:
# === Cell 1: setup and editable training config ===
import importlib
from datetime import datetime

import pandas as pd
import torch

import recursive_corepromoter_design as legacy
import tss_pas_dataset as tss_data
import train_corepromoter_tss_pas as trainer
import evaluate_corepromoter_tss_pas as evaluator

importlib.reload(legacy)
importlib.reload(tss_data)
importlib.reload(trainer)
importlib.reload(evaluator)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RUN_STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
DATASET_DIR = legacy.PROJECT_ROOT / "outputs" / "tss_pas_dataset"
DATASET_PATH = DATASET_DIR / "tss_pas_processed.pkl"
RUN_DIR = legacy.PROJECT_ROOT / "outputs" / "corepromoter_tss_pas" / RUN_STAMP
BASELINE_CHECKPOINT = legacy.WEIGHTS_DIR / "weights_CorePromoter_clean.pt"
ARCH_CHECKPOINT = legacy.WEIGHTS_DIR / "weights_CorePromoter_tss_arch.pt"
FINAL_CHECKPOINT = legacy.WEIGHTS_DIR / "weights_CorePromoter_tss_pas.pt"

TRAINING_CONFIG = trainer.TrainingConfig(
    batch_size=512,
    arch_epochs=20,
    head_epochs=15,
    joint_epochs=15,
)

RUN_STRICT_LOLO = False  # Expensive: retrains both models for all seven held-out libraries.
print("Device:", DEVICE)
print("Run directory:", RUN_DIR)
print("Final checkpoint:", FINAL_CHECKPOINT)

In [ ]:
# === Cell 2: prepare and verify the TSS/PAS dataset ===
dataset_outputs = tss_data.save_processed_dataset(
    xlsx_path=tss_data.DEFAULT_XLSX,
    output_dir=DATASET_DIR,
    validation_fraction=0.15,
    test_fraction=0.15,
    seed=legacy.SEED,
    allow_n=True,
)
display(pd.read_csv(dataset_outputs["qc_summary"]))
display(pd.read_csv(dataset_outputs["qc_failure_summary"]).head(20))
display(pd.read_csv(dataset_outputs["split_summary"]))
display(pd.read_csv(dataset_outputs["split_groups"]))
print("Processed dataset:", dataset_outputs["dataset"])

In [ ]:
# === Cell 3: train once and save frozen checkpoints ===
training_outputs = trainer.train_production_model(
    dataset_path=DATASET_PATH,
    baseline_checkpoint=BASELINE_CHECKPOINT,
    arch_checkpoint=ARCH_CHECKPOINT,
    final_checkpoint=FINAL_CHECKPOINT,
    run_dir=RUN_DIR,
    device=DEVICE,
    config=TRAINING_CONFIG,
)
display(pd.read_csv(training_outputs["metrics"]))
print("Architecture checkpoint:", training_outputs["arch_checkpoint"])
print("Final recursive checkpoint:", training_outputs["final_checkpoint"])

In [ ]:
# === Cell 4: compare the saved baseline and TSS/PAS checkpoints ===
processed_tss = pd.read_pickle(DATASET_PATH)
expression_data = legacy.build_core_training_dataframe()
comparison = evaluator.checkpoint_comparison(
    BASELINE_CHECKPOINT,
    FINAL_CHECKPOINT,
    processed_tss,
    expression_data,
    DEVICE,
    TRAINING_CONFIG.batch_size,
)
comparison_path = RUN_DIR / "checkpoint_comparison_metrics.csv"
comparison.to_csv(comparison_path, index=False)
display(comparison)
print("Saved:", comparison_path)

In [ ]:
# === Cell 5: optional strict leave-one-library-out comparison ===
# This is intentionally off by default because it retrains 14 fold models.
if RUN_STRICT_LOLO:
    lolo_metrics, lolo_history = evaluator.leave_one_library_out_comparison(
        expression_data,
        processed_tss,
        DEVICE,
        TRAINING_CONFIG,
        baseline_epochs=50,
    )
    lolo_metrics.to_csv(RUN_DIR / "leave_one_library_out_metrics.csv", index=False)
    lolo_history.to_csv(RUN_DIR / "leave_one_library_out_history.csv", index=False)
    display(lolo_metrics)
else:
    print("Skipped strict LOLO. Set RUN_STRICT_LOLO=True when needed.")

## Next step

Open `Model_CorePromoter_recursive_design.ipynb`, keep `CORE_MODEL_VARIANT = "tss_pas"`, and run the existing recursive design cells. The recursive notebook loads the frozen `weights_CorePromoter_tss_pas.pt`; it does not retrain the neural network.